# Backtest Explorer

**Pick a stock and a time window, and see how the breakout strategy would have traded it.**

This notebook replays history one day at a time (no look-ahead), takes every breakout signal, manages each trade (scale-out + trailing stop + time decay), and shows you the result: a trade log, a price chart with entries/exits, and an equity curve vs buy-and-hold.

Edit the **Parameters** cell, then *Run All*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from breakout.config import Settings
from breakout.data.loader import load_or_fetch
from breakout.backtest.engine import BacktestConfig, backtest_symbol

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True

## 1. Parameters

- `TICKER` — the stock to test.
- `START` / `END` — the test window (`None` = use the trailing `LOOKBACK_DAYS`).
- Bars before `START` are used as warm-up for the moving averages, so fetch generously.

In [ ]:
TICKER        = 'SNDK'
BENCHMARK     = 'SPY'
START         = None          # e.g. '2024-01-01'  (None -> trailing window)
END           = None          # e.g. '2024-12-31'
LOOKBACK_DAYS = 365           # length of the test window when START/END are None
EQUITY        = 100_000.0
MIN_COMPOSITE = 0.0           # only take signals at/above this composite rank
FETCH_DAYS    = 730           # how much history to pull (test window + warm-up)

## 2. Load bars

Reads the local cache; downloads from Alpaca (and caches) if missing.

In [ ]:
df    = load_or_fetch(TICKER, lookback_days=FETCH_DAYS)
bench = load_or_fetch(BENCHMARK, lookback_days=FETCH_DAYS)
print(f'{TICKER}: {len(df)} bars  {df.index[0].date()} -> {df.index[-1].date()}')
print(f'{BENCHMARK}: {len(bench)} bars')
df.tail()

## 3. Run the backtest

In [ ]:
settings = Settings.load()
config = BacktestConfig.from_settings(settings, account_equity=EQUITY)
config.min_composite = MIN_COMPOSITE

res = backtest_symbol(TICKER, df, bench, settings, config, start=START, end=END)

w0, w1 = res.window
m, e = res.metrics, res.equity
print(f'Window      : {w0.date()} -> {w1.date()}  ({e.get("n_bars", 0)} bars)')
print(f'Signals     : {res.n_signals}')
print(f'Trades      : {m.get("n", 0)}')
if m.get('n', 0):
    print()
    print(f'Win rate     : {m["win_rate"]*100:.1f}%')
    print(f'Expectancy   : {m["expectancy_r"]:+.2f} R / trade')
    print(f'Avg win/loss : {m["avg_win_r"]:+.2f}R / {m["avg_loss_r"]:+.2f}R')
    print(f'Profit factor: {m["profit_factor"]}')
    print(f'Total        : {m["total_r"]:+.2f}R')
if e.get('n_bars', 0):
    print()
    print(f'Return       : {e["total_return_pct"]:+.2f}%  (${e["start_equity"]:,.0f} -> ${e["end_equity"]:,.0f})')
    print(f'Max drawdown : {e["max_drawdown_pct"]:.2f}%')
    verdict = 'BEAT' if e['total_return_pct'] > res.buy_hold_pct else 'TRAILED'
    print(f'Buy & hold   : {res.buy_hold_pct:+.2f}%  -> strategy {verdict} buy-and-hold')

## 4. Trade log

In [ ]:
rows = [{
    'entry_date': t.entry_date.date(), 'exit_date': t.exit_date.date(),
    'entry': round(t.entry, 2), 'exit': round(t.exit, 2),
    'shares': t.shares, 'bars_held': t.bars_held,
    'R': round(t.r_multiple, 2), 'reason': t.exit_reason,
} for t in res.trades]
trades_df = pd.DataFrame(rows)
trades_df

## 5. Price chart with entries & exits

Green ▲ = entry, red ▼ = exit. The dashed line under each trade is its initial stop.

In [ ]:
win = df.loc[w0:w1]
fig, ax = plt.subplots()
ax.plot(win.index, win['close'], color='#444', lw=1.2, label=f'{TICKER} close')
for t in res.trades:
    color = '#16a34a' if t.r_multiple > 0 else '#dc2626'
    ax.scatter(t.entry_date, t.entry, marker='^', s=120, color='#16a34a', zorder=5)
    ax.scatter(t.exit_date, t.exit, marker='v', s=120, color='#dc2626', zorder=5)
    ax.plot([t.entry_date, t.exit_date], [t.entry, t.exit], color=color, lw=1, alpha=.6)
    ax.hlines(t.stop, t.entry_date, t.exit_date, color=color, ls='--', lw=.8, alpha=.5)
ax.set_title(f'{TICKER}  {w0.date()} -> {w1.date()}  —  entries (^) and exits (v)')
ax.legend(loc='upper left')
plt.show()

## 6. Equity curve vs buy-and-hold

Both start at the same equity. The strategy sits in cash between trades, so on a stock that only goes up, buy-and-hold wins. The strategy earns its keep on **choppy** names by avoiding the drawdowns — watch the gap during selloffs.

In [ ]:
eq = res.equity_curve
bh = win['close'] / float(win['close'].iloc[0]) * EQUITY   # buy & hold, same start
fig, ax = plt.subplots()
ax.plot(eq.index, eq.values, color='#2563eb', lw=1.6, label='strategy equity')
ax.plot(bh.index, bh.values, color='#9ca3af', lw=1.4, ls='--', label=f'buy & hold {TICKER}')
ax.set_title('Equity curve vs buy-and-hold')
ax.set_ylabel('account value ($)')
ax.legend(loc='upper left')
plt.show()

---
**How to read this:** the headline number is **expectancy (R/trade)** — positive means a real edge. `R` is profit in risk units (+2R = made twice what it risked; losers should sit near −1R). Try other tickers, widen the window, or raise `MIN_COMPOSITE` to take only the strongest setups.